# 01 — Análise Exploratória de Dados (EDA)

**Dataset:** Vehicle Collision Data in Seattle (2005–2019)  
**Tarefa:** Classificação da severidade dos acidentes (`SEVERITYCODE`)

---

## Estrutura deste notebook

1. Carregamento e inspeção geral
2. Análise da variável alvo
3. Análise univariada (numéricas e categóricas)
4. Análise bivariada (features × target)
5. Matriz de correlação
6. Conclusões parciais

## 1. Carregamento e Inspeção Geral

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

df = pd.read_csv("data/seattle_collision_data_2005_2019.csv")

print("Shape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)


In [ ]:
df.head(5)

In [ ]:
nulos = pd.DataFrame({
    "ausentes": df.isnull().sum(),
    "percentual (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos["ausentes"] > 0].sort_values("percentual (%)", ascending=False)
print(nulos)


In [ ]:
cols_sem_id = [c for c in df.columns if c not in ["Unnamed: 0", "SPDCASENO"]]
dup = df[cols_sem_id].duplicated().sum()
print(f"Linhas duplicadas (excluindo colunas de ID): {dup}")
print(f"  → {dup / len(df) * 100:.2f}% do total de {len(df)} registros\n")

df.describe().T


## 2. Análise da Variável Alvo — `SEVERITYCODE`

O target é `SEVERITYCODE`, que representa a severidade da colisão.  
É fundamental entender a **distribuição das classes** antes de qualquer modelagem:
- Desbalanceamento entre classes pode enviesar o modelo e exige estratégias específicas (ex: `class_weight='balanced'`, oversampling).

In [ ]:
contagem = df["SEVERITYCODE"].value_counts().sort_index()
proporcao = df["SEVERITYCODE"].value_counts(normalize=True).sort_index() * 100

resumo_target = pd.DataFrame({"contagem": contagem, "proporção (%)": proporcao.round(2)})
print(resumo_target)

baseline_acc = proporcao.max()
print(f"\nBaseline de accuracy (prever sempre a classe majoritária): {baseline_acc:.2f}%")
print("→ Qualquer modelo deve superar esse valor para ser útil.")

fig, ax = plt.subplots()
bars = ax.bar(resumo_target.index.astype(str), resumo_target["contagem"],
              color=sns.color_palette("Blues_d", len(resumo_target)))
ax.set_xlabel("SEVERITYCODE")
ax.set_ylabel("Quantidade")
ax.set_title("Distribuição da variável alvo — SEVERITYCODE")

for bar, (_, row) in zip(bars, resumo_target.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
            f"{row['proporção (%)']:.1f}%", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()


## 3. Análise Univariada

### 3.1 Variáveis Numéricas

Histogramas mostram a **distribuição de frequência** de cada variável.  
Observamos assimetria (skewness), presença de outliers e a forma geral da distribuição.

In [ ]:
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()

LEAKAGE = ["INJURIES", "SERIOUSINJURIES", "FATALITIES"]
EXCLUIR  = ["Unnamed: 0", "SEVERITYCODE", "LIGHTCOND"] + LEAKAGE

colunas_numericas = [c for c in colunas_numericas if c not in EXCLUIR]

print("Variáveis numéricas para análise:", colunas_numericas)
print(f"\n⚠ Excluídas por leakage: {LEAKAGE}")
print("  → São consequências do acidente e não podem ser usadas como preditores.\n")

n = len(colunas_numericas)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(colunas_numericas):
    axes[i].hist(df[col].dropna(), bins=30, color="#4C72B0", edgecolor="white")
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribuição das variáveis numéricas (sem leakage)", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

print("Assimetria (skewness) das variáveis numéricas:")
print(df[colunas_numericas].skew().sort_values(ascending=False).round(3))


### 3.2 Variáveis Categóricas

Barplots mostram a frequência de cada categoria.  
Categorias com frequência muito baixa podem ser agrupadas em "outros" para reduzir dimensionalidade após encoding.

In [ ]:
colunas_categoricas = df.select_dtypes(include=["str", "object"]).columns.tolist()
colunas_categoricas = [c for c in colunas_categoricas if c not in ["SPDCASENO", "DATE"]]

colunas_bool = df.select_dtypes(include=["bool"]).columns.tolist()

df["LIGHTCOND_cat"] = df["LIGHTCOND"].astype(str)

todas_categoricas = colunas_categoricas + ["LIGHTCOND_cat"] + colunas_bool
print("Variáveis categóricas analisadas:", todas_categoricas)

fig, axes = plt.subplots(len(todas_categoricas), 1, figsize=(12, len(todas_categoricas) * 4))

for i, col in enumerate(todas_categoricas):
    ordem = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=ordem, ax=axes[i], hue=col, palette="Blues_d", legend=False)
    axes[i].set_title(f"Distribuição — {col}", fontsize=10)
    axes[i].set_xlabel("Contagem")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.show()

print("\nCardinalidade:")
for col in todas_categoricas:
    print(f"  {col}: {df[col].nunique()} categorias únicas")


### 3.3 Variáveis Temporais — DATE e TIME

DATE e TIME não entram diretamente como features, mas podem esconder padrões importantes:
hora do dia, mês e dia da semana podem influenciar a severidade dos acidentes.
Aqui apenas observamos os padrões — a extração dessas features ocorre no pré-processamento.

In [ ]:
df_temp = df.copy()
df_temp["DATE"] = pd.to_datetime(df_temp["DATE"], errors="coerce")
df_temp["mes"]        = df_temp["DATE"].dt.month
df_temp["dia_semana"] = df_temp["DATE"].dt.dayofweek
df_temp["hora"]       = (df_temp["TIME"].dropna() // 100).astype("Int64")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df_temp["mes"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].set_title("Acidentes por mês")
axes[0].set_xlabel("Mês")
axes[0].set_ylabel("Contagem")

dias = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
df_temp["dia_semana"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#4C72B0", edgecolor="white")
axes[1].set_xticklabels(dias, rotation=0)
axes[1].set_title("Acidentes por dia da semana")
axes[1].set_xlabel("Dia")

df_temp["hora"].value_counts().sort_index().plot(kind="bar", ax=axes[2], color="#4C72B0", edgecolor="white")
axes[2].set_title("Acidentes por hora do dia")
axes[2].set_xlabel("Hora")

plt.tight_layout()
plt.show()

print("Severidade média (SEVERITYCODE) por hora do dia:")
print(df_temp.groupby("hora")["SEVERITYCODE"].mean().round(3).to_string())


## 4. Análise Bivariada — Features × Target

Analisamos como cada variável se relaciona com `SEVERITYCODE`.  
Para variáveis categóricas: proporção de cada classe do target por categoria (stacked barplot normalizado).  
Para variáveis numéricas: boxplot por classe do target — permite identificar diferenças de mediana, dispersão e outliers entre as classes.

In [ ]:
for col in todas_categoricas:
    tabela = pd.crosstab(df[col], df["SEVERITYCODE"], normalize="index") * 100
    contagem_cat = df[col].value_counts()
    categorias_validas = contagem_cat[contagem_cat >= 50].index
    tabela = tabela.loc[tabela.index.isin(categorias_validas)]

    tabela.plot(kind="bar", stacked=True, figsize=(10, 4), colormap="Blues")
    plt.title(f"{col} × SEVERITYCODE (proporção %)")
    plt.xlabel(col)
    plt.ylabel("Proporção (%)")
    plt.legend(title="SEVERITYCODE", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()


In [ ]:
features_bivar = [c for c in colunas_numericas if c not in ["response_time", "SNOW", "SNWD"]]

ncols = 4
nrows = (len(features_bivar) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()

for i, col in enumerate(features_bivar):
    sns.boxplot(data=df, x="SEVERITYCODE", y=col, ax=axes[i],
                palette="Blues", hue="SEVERITYCODE", legend=False)
    axes[i].set_title(col, fontsize=9)
    axes[i].set_xlabel("SEVERITYCODE")
    axes[i].set_ylabel("")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Variáveis numéricas por classe de SEVERITYCODE", fontsize=12)
plt.tight_layout()
plt.show()


## 5. Matriz de Correlação

A correlação de Pearson mede a **relação linear** entre variáveis numéricas (valores entre -1 e 1).  
- Valores próximos de ±1: forte correlação linear  
- Valores próximos de 0: fraca ou nenhuma correlação linear  

Identificar features altamente correlacionadas entre si é importante para evitar **multicolinearidade**, que pode prejudicar modelos como a Regressão Logística.

In [ ]:
colunas_corr = colunas_numericas + ["SEVERITYCODE"]
corr_matrix = df[colunas_corr].corr(method="pearson")

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    annot_kws={"size": 7}
)
plt.title("Matriz de Correlação de Pearson — variáveis numéricas", fontsize=12)
plt.tight_layout()
plt.show()

print("Correlação de Pearson com SEVERITYCODE (ordenada por valor absoluto):")
print(corr_matrix["SEVERITYCODE"].drop("SEVERITYCODE").sort_values(key=abs, ascending=False).round(3))

print("\nPares com alta correlação entre si (|r| > 0.8) — risco de multicolinearidade:")
threshold = 0.8
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > threshold:
            print(f"  {corr_matrix.columns[i]} × {corr_matrix.columns[j]}: r = {val:.3f}")


### 5.2 Informação Mútua — Features numéricas × Target

A **Informação Mútua (MI)** mede a dependência entre variável e target sem assumir linearidade,
sendo mais adequada para targets multi-classe.
Valores maiores indicam maior relevância da feature para classificação.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_mi = df[colunas_numericas].fillna(df[colunas_numericas].median(numeric_only=True))
y_mi = df["SEVERITYCODE"]

mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=colunas_numericas).sort_values(ascending=False)

print("Informação Mútua com SEVERITYCODE (maior = mais relevante):")
print(mi_series.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
mi_series.plot(kind="bar", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Informação Mútua — features × SEVERITYCODE")
ax.set_ylabel("MI Score")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Conclusões Parciais da EDA

Registre aqui os principais achados após executar o notebook:

In [ ]:
conclusoes = """
CONCLUSÕES PARCIAIS — EDA
=========================

1. DISTRIBUIÇÃO DO TARGET (SEVERITYCODE):
   - 4 classes: 0 (64.33%), 1 (33.79%), 2 (1.71%), 3 (0.17%)
   - Dataset fortemente desbalanceado — classes 2 e 3 somam < 2%
   - Baseline de accuracy (prever sempre classe 0) = 64.33%
   - Implicação: usar class_weight='balanced'; métrica principal = F1-macro

2. VALORES AUSENTES:
   - response_type / response_time: 86.55% → remover no pré-processamento
   - SNOW / SNWD: 24.19% → imputar com 0 (ausência de neve é informativa)
   - WSF5: 1.21% → imputar com mediana

3. DUPLICATAS:
   - 19 linhas duplicadas (excluindo colunas de ID) → remover no pré-processamento

4. LEAKAGE — excluídas da modelagem:
   - INJURIES, SERIOUSINJURIES, FATALITIES: são resultados do acidente, não preditores

5. MULTICOLINEARIDADE (pares com |r| > 0.8):
   - TAVG × TMAX: r = 0.969 → remover TMAX e TMIN, manter TAVG
   - TAVG × TMIN: r = 0.954
   - TMAX × TMIN: r = 0.875
   - AWND × WSF5: r = 0.814 → remover WSF5, manter AWND
   - SNOW × SNWD: verificar no pré-processamento após imputação

6. FEATURES MAIS RELEVANTES (Informação Mútua):
   - VEHCOUNT:    0.0823  (nº de veículos)
   - response_time: 0.0476 (mas tem 86% nulos — será removida)
   - PEDCOUNT:    0.0427  (pedestres envolvidos)
   - longitude:   0.0408  (localização geográfica tem padrão)
   - latitude:    0.0404
   - Variáveis climáticas têm MI próximo de 0 — pouco poder preditivo

7. ENCODING NO PRÉ-PROCESSAMENTO:
   - Categóricas nominais (COLLISIONTYPE, WEATHER, ROADCOND, JUNCTIONTYPE) → One-Hot Encoding
   - LIGHTCOND (ordinal) → manter como inteiro ordinal
   - Booleanas (SPEEDING, UNDERINFL, INATTENTIONIND, HITPARKEDCAR, intersection_related) → int (0/1)
"""
print(conclusoes)
